Found XSRF token path from HAR file of a request to trains api: https://eticket.railway.uz/api/v3/handbook/trains/list


In [10]:
import json

with open("full_request.har", encoding="utf-8") as f:
    har = json.load(f)

for entry in har["log"]["entries"]:
    for header in entry["response"]["headers"]:
        if "xsrf" in header["value"].lower() or "xsrf" in header["name"].lower():
            print(entry["request"]["method"], entry["request"]["url"])
            print(f"  {header['name']}: {header['value']}")
            print()
            break

GET https://eticket.railway.uz/api/v1/csrf-token
  set-cookie: XSRF-TOKEN=b78ca210-a9c4-4c43-a9b9-bf95d520109b; Path=/



Testing token extraction.

In [11]:
import requests

session = requests.Session()
session.get("https://eticket.railway.uz/api/v1/csrf-token")
csrf_token = session.cookies.get("XSRF-TOKEN")
print(csrf_token)

cc09c9d7-b8d6-4094-92d5-1e5f81b7d9d1


Now make a test request to trains API with the token.

In [ ]:
import requests
import json

session = requests.Session()

# 1. Get CSRF token
session.get("https://eticket.railway.uz/api/v1/csrf-token")
csrf_token = session.cookies.get("XSRF-TOKEN")

# 2. Query trains
resp = session.post(
    "https://eticket.railway.uz/api/v3/handbook/trains/list",
    headers={
        "X-XSRF-TOKEN": csrf_token,
        "Content-Type": "application/json",
        "Accept": "application/json",
        "Accept-Language": "ru",
        "Referer": "https://eticket.railway.uz/ru/home",
        "Origin": "https://eticket.railway.uz",
        "device-type": "BROWSER",
    },
    json={
        "directions": {
            "forward": {
                "date": "2026-04-29",
                "depStationCode": "2900000",
                "arvStationCode": "2900700",
            }
        }
    },
)

print(resp.status_code)
print(json.dumps(resp.json(), indent=2, ensure_ascii=False))

The brand field is cleanest binning key. Across both responses:

Afrosiyob → type varies inconsistently: "Высокоскоростной", "скрст", "СКРСТ" — but brand is always "Afrosiyob"

Sharq → brand: "Sharq", type "СК"

Nasaf → brand: "Nasaf", type "СК"

Everything else → brand is "Пассажирский", "скорый", "Cкорый" (note the Latin C in one of them!)

Departure stations. "depStationName": 
"ТАШКЕНТ": "2900000",
"САМАРКАНД": "2900700"
"БУХАРА": "2900800"

In [ ]:
BINS = {
    "Afrosiyob": "high_speed",
    "Sharq": "fast",
    "Nasaf": "fast",
}

def bin_train(train):
    return BINS.get(train["brand"], "basic")